In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torchvision import datasets, transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
SOURCE_DATASET = "/path/to/original_dataset"
OUTPUT_DATASET = "/path/to/split_dataset"

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

In [ ]:
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DATASET, split), exist_ok=True)

In [ ]:
for class_name in os.listdir(SOURCE_DATASET):

    class_path = os.path.join(SOURCE_DATASET, class_name)
    images = os.listdir(class_path)
    random.shuffle(images)

    n = len(images)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    splits = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split, files in splits.items():
        dest_dir = os.path.join(OUTPUT_DATASET, split, class_name)
        os.makedirs(dest_dir, exist_ok=True)

        for f in files:
            shutil.copy(
                os.path.join(class_path, f),
                os.path.join(dest_dir, f)
            )

In [ ]:
DATASET_PATH = "/path/to/Datasplit_5"

train_dir = os.path.join(DATASET_PATH, "Train")
valid_dir = os.path.join(DATASET_PATH, "Valid")
test_dir  = os.path.join(DATASET_PATH, "Test")

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
valid_dataset = datasets.ImageFolder(valid_dir, transform=test_transform)
test_dataset  = datasets.ImageFolder(test_dir,  transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)

num_classes = len(train_dataset.classes)
print("Classes:", train_dataset.classes)

In [ ]:
class ClassBalancedFocalLoss(nn.Module):

    def __init__(self, samples_per_class, num_classes, beta=0.9999, gamma=2.0):
        super().__init__()

        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / effective_num
        weights = weights / np.sum(weights) * num_classes

        self.weights = torch.tensor(weights).float().to(device)
        self.gamma = gamma

    def forward(self, logits, labels):

        ce_loss = nn.CrossEntropyLoss(weight=self.weights)(logits, labels)
        pt = torch.exp(-ce_loss)

        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss

In [ ]:
weights = ViT_B_16_Weights.DEFAULT
model = vit_b_16(weights=weights)

model.heads.head = nn.Linear(model.heads.head.in_features, 5)

model = model.to(device)

In [ ]:
samples_per_class = [len(os.listdir(os.path.join(train_dir, c))) for c in train_dataset.classes]

criterion = ClassBalancedFocalLoss(samples_per_class, num_classes)

optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def train_epoch(loader):

    model.train()
    total_loss = 0
    correct = 0

    for images, labels in tqdm(loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, 1)
        correct += (preds == labels).sum().item()

    acc = correct / len(loader.dataset)

    return total_loss, acc

In [ ]:
def evaluate(loader):

    model.eval()
    total_loss = 0
    correct = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = torch.argmax(outputs,1)
            correct += (preds == labels).sum().item()

    acc = correct / len(loader.dataset)

    return total_loss, acc

In [ ]:
EPOCHS = 30

for epoch in range(EPOCHS):

    train_loss, train_acc = train_epoch(train_loader)

    val_loss, val_acc = evaluate(valid_loader)

    print(f"Epoch {epoch+1}")

    print("Train Loss:", train_loss)
    print("Train Acc:", train_acc)

    print("Val Loss:", val_loss)
    print("Val Acc:", val_acc)